# Model Experiments & Hyperparameter Tuning

This notebook trains multiple machine learning classifiers on the processed COMPAS dataset and performs GridSearchCV hyperparameter tuning.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

from configs.config import RANDOM_STATE, TEST_SIZE, TARGET_COLUMN, PROCESSED_PATH
from src.models.model_factory import ModelFactory
from src.models.hyperparameter_tuning import tune_model
from src.evaluation.metrics import evaluate_predictions

### Load the Processed Extended Dataset

In [ ]:
data_path = Path("../") / PROCESSED_PATH / "compas_extended.csv"
df = pd.read_csv(data_path)
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

### Train Base Classifiers and Tune Hyperparameters

In [ ]:
models_list = ["logistic_regression", "decision_tree", "random_forest", "xgboost"]
results = []

for model_name in models_list:
    print(f"Tuning {model_name}...")
    base_model = ModelFactory.get_model(model_name, random_state=RANDOM_STATE)
    best_estimator, best_params = tune_model(
        model_name=model_name,
        base_estimator=base_model,
        X=X_train,
        y=y_train,
        search_type="grid",
        cv=3
    )
    
    # Evaluate on test set
    y_pred = best_estimator.predict(X_test)
    y_prob = best_estimator.predict_proba(X_test)[:, 1] if hasattr(best_estimator, "predict_proba") else None
    perf = evaluate_predictions(y_test, y_pred, y_prob)
    
    results.append({
        "model_name": model_name,
        "best_params": best_params,
        **perf
    })

df_results = pd.DataFrame(results)
df_results